<a href="https://colab.research.google.com/github/hiteshraos/BIS-LAB/blob/main/1BM24CS117_CHARGING_STATION_PCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from concurrent.futures import ThreadPoolExecutor
import copy
import math


class ChargingStationCell:

    def __init__(self, r, c, capacity):
        self.r = r
        self.c = c
        self.capacity = capacity
        self.evs = []  # List of dicts: {'id': str, 'energy': float}

    def get_cost(self):
        if self.capacity == 0:
            return float('inf')
        total_energy = sum(ev['energy'] for ev in self.evs)
        queue_penalty = max(0, len(self.evs) - self.capacity) * 10
        return (total_energy / self.capacity) + queue_penalty


class ParallelCellularEVOptimizer:

    def __init__(self, rows, cols, capacities, travel_weight=2.5):
        self.rows = rows
        self.cols = cols
        self.travel_weight = travel_weight
        self.grid = [
            [
                ChargingStationCell(r, c, capacities[r][c])
                for c in range(cols)
            ]
            for r in range(rows)
        ]

    def assign_initial_evs(self, ev_list):
        for ev in ev_list:
            r, c = ev['target_r'], ev['target_c']
            self.grid[r][c].evs.append(ev)

    def get_neighbors(self, r, c):
        neighbors = []
        for dr in [-1, 0, 1]:
            for dc in [-1, 0, 1]:
                if dr == 0 and dc == 0:
                    continue
                nr, nc = r + dr, c + dc
                if 0 <= nr < self.rows and 0 <= nc < self.cols:
                    dist = math.hypot(dr, dc)
                    neighbors.append((self.grid[nr][nc], dist))
        return neighbors

    def evaluate_cell_migration(self, cell):
        """Parallel worker function to determine if a cell should offload an EV."""
        if len(cell.evs) <= cell.capacity:
            return None  # Station is not overloaded

        current_cost = cell.get_cost()
        best_neighbor = None
        min_migration_cost = current_cost

        for neighbor, dist in self.get_neighbors(cell.r, cell.c):
            migration_cost = neighbor.get_cost() + (self.travel_weight * dist)
            if migration_cost < min_migration_cost:
                min_migration_cost = migration_cost
                best_neighbor = neighbor

        if best_neighbor and cell.evs:
            # Select the last EV in queue to migrate
            ev_to_move = cell.evs[-1]
            return (cell.r, cell.c, best_neighbor.r, best_neighbor.c, ev_to_move)
        return None

    def optimize(self, max_iterations=10):
        for iteration in range(1, max_iterations + 1):
            migrations = []
            all_cells = [
                self.grid[r][c]
                for r in range(self.rows)
                for c in range(self.cols)
            ]

            # Step 1: Parallel Evaluation
            with ThreadPoolExecutor() as executor:
                results = list(
                    executor.map(self.evaluate_cell_migration, all_cells)
                )

            # Collect valid migrations
            for res in results:
                if res is not None:
                    migrations.append(res)

            # Stop early if equilibrium is reached
            if not migrations:
                print(
                    f'\n[Equilibrium Reached at Iteration {iteration - 1}]'
                )
                break

            # Step 2: Synchronous State Update
            for src_r, src_c, dst_r, dst_c, ev in migrations:
                if ev in self.grid[src_r][src_c].evs:
                    self.grid[src_r][src_c].evs.remove(ev)
                    self.grid[dst_r][dst_c].evs.append(ev)

    def display_grid(self, title):
        print(f'\n--- {title} ---')
        for r in range(self.rows):
            row_str = []
            for c in range(self.cols):
                cell = self.grid[r][c]
                ev_ids = [e['id'] for e in cell.evs]
                row_str.append(
                    f'Station({r},{c}) [Cap:{cell.capacity} | Cost:{cell.get_cost():.1f}]: {ev_ids}'
                )
            print('  |  '.join(row_str))


# ==========================================
# MAIN EXECUTION (WITH ROBUST USER FALLBACKS)
# ==========================================
if __name__ == '__main__':
    print('=== EV Charging Station Allocation (Parallel Cellular Algorithm) ===')

    # Default values to fall back on if user inputs are skipped or invalid
    default_rows = 2
    default_cols = 2
    default_capacities = [[5, 4], [8, 1]]
    default_evs = [
        {'id': 'EV1', 'target_r': 0, 'target_c': 1, 'energy': 50.0},
        {'id': 'EV2', 'target_r': 1, 'target_c': 0, 'energy': 40.0},
        {'id': 'EV3', 'target_r': 0, 'target_c': 0, 'energy': 30.0},
        {'id': 'EV4', 'target_r': 0, 'target_c': 0, 'energy': 20.0}
    ]
    default_iterations = 5

    try:
        use_defaults = input('Use default mock configurations? (y/n) [Default: y]: ').strip().lower() != 'n'
    except (KeyboardInterrupt, SystemExit, Exception):
        use_defaults = True

    if use_defaults:
        rows = default_rows
        cols = default_cols
        capacities = default_capacities
        ev_list = default_evs
        iterations = default_iterations
        print('\nUsing preset parameters for demonstrations.')
    else:
        try:
            rows = int(input('Enter grid rows (e.g., 2): ') or default_rows)
            cols = int(input('Enter grid columns (e.g., 2): ') or default_cols)

            print('\nEnter charger capacity (number of ports) for each station in the grid:')
            capacities = []
            for r in range(rows):
                row_input = input(f'Row {r} capacities (space-separated {cols} integers): ').split()
                if not row_input:
                    row_cap = default_capacities[r] if r < len(default_capacities) else [3] * cols
                else:
                    row_cap = list(map(int, row_input))
                capacities.append(row_cap)

            num_evs_input = input('\nEnter total number of incoming EVs: ')
            num_evs = int(num_evs_input) if num_evs_input else len(default_evs)

            ev_list = []
            print('Enter EV details -> ID, Initial Target Row, Initial Target Col, Energy Needed (kWh):')
            for i in range(num_evs):
                raw_ev = input(f'EV {i+1} (e.g., EV1 0 0 40): ').split()
                if len(raw_ev) < 4:
                    print("Invalid input, using a default EV instead.")
                    ev_list.append(default_evs[i % len(default_evs)])
                else:
                    ev_list.append({
                        'id': raw_ev[0],
                        'target_r': int(raw_ev[1]),
                        'target_c': int(raw_ev[2]),
                        'energy': float(raw_ev[3]),
                    })

            iter_input = input('\nEnter max iterations (e.g., 5): ')
            iterations = int(iter_input) if iter_input else default_iterations
        except Exception as e:
            print(f"\nEncountered error during manual setup: {e}. Falling back to default configuration.")
            rows = default_rows
            cols = default_cols
            capacities = default_capacities
            ev_list = default_evs
            iterations = default_iterations

    optimizer = ParallelCellularEVOptimizer(rows, cols, capacities)
    optimizer.assign_initial_evs(ev_list)

    optimizer.display_grid('INITIAL UNBALANCED ALLOCATION')
    optimizer.optimize(max_iterations=iterations)
    optimizer.display_grid('OPTIMIZED BALANCED ALLOCATION')

=== EV Charging Station Allocation (Parallel Cellular Algorithm) ===
Use default mock configurations? (y/n) [Default: y]: N
Enter grid rows (e.g., 2): 2
Enter grid columns (e.g., 2): 2

Enter charger capacity (number of ports) for each station in the grid:
Row 0 capacities (space-separated 2 integers): 50 60
Row 1 capacities (space-separated 2 integers): 70 100

Enter total number of incoming EVs: 4
Enter EV details -> ID, Initial Target Row, Initial Target Col, Energy Needed (kWh):
EV 1 (e.g., EV1 0 0 40): EV1 0 0 40
EV 2 (e.g., EV1 0 0 40): EV2 0 0 50
EV 3 (e.g., EV1 0 0 40): EV3 1 0 20
EV 4 (e.g., EV1 0 0 40): EV4 1 0 10

Enter max iterations (e.g., 5): 5

--- INITIAL UNBALANCED ALLOCATION ---
Station(0,0) [Cap:50 | Cost:1.8]: ['EV1', 'EV2']  |  Station(0,1) [Cap:60 | Cost:0.0]: []
Station(1,0) [Cap:70 | Cost:0.4]: ['EV3', 'EV4']  |  Station(1,1) [Cap:100 | Cost:0.0]: []

[Equilibrium Reached at Iteration 0]

--- OPTIMIZED BALANCED ALLOCATION ---
Station(0,0) [Cap:50 | Cost:1.8]: ['